# Validate Content

Validates standardized Work content before promotion to Silver. The notebook verifies document identity, content availability, and canonical ID uniqueness, identifies valid records for Silver promotion, reconciles Bronze source files with validated target records, and publishes trusted content documents to the Silver layer.

## Input Parameters

In [ ]:
p_source_container = "DataOps_Control"
p_source_execution_step_id = 116
p_execution_step_id = 0
p_created_by = ""

## Imports

In [ ]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F

## Shared Validation Helpers

Load `get_validation_condition()` and `add_validation_result()` from `00_nb_validation_helpers` in this Fabric workspace. The shared notebook defines no execution state.

In [ ]:
%run 00_nb_validation_helpers

## Configuration

Defines the file types supported by the current content-validation and reconciliation process.

In [ ]:
supported_extensions = [
    ".md",
    ".sql",
    ".txt"
]

## Load Work Content

Loads the current content candidates from the Work layer for the selected source container.

In [ ]:
def load_work_content(
    source_container
):
    """
    Load the current Work content candidates for the selected
    source container.
    """

    return (
        spark.table("work.content_documents")
        .filter(
            F.col("source_container") == source_container
        )
    )

## Validate Content Documents

Applies row-level validation rules to Work content documents before Silver promotion.

In [ ]:
def validate_required_content_id(
    work_content_documents,
    validation_results
):
    """
    Validate that every content document has a canonical content_id.
    """

    work_content_documents = (
        work_content_documents
        .withColumn(
            "is_content_id_not_null_passed",
            F.col("content_id").isNotNull()
        )
    )

    affected_row_count = (
        work_content_documents
        .filter(~F.col("is_content_id_not_null_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Required ID validation: "
            "CONTENT_DOCUMENTS.content_id contains null values."
        ),
        affected_row_count=affected_row_count,
        validation_code="NOT_NULL"
    )

    return work_content_documents

In [ ]:
def validate_required_content(
    work_content_documents,
    validation_results
):
    """
    Validate that every content document contains usable textual content.
    """

    work_content_documents = (
        work_content_documents
        .withColumn(
            "is_content_present_passed",
            (
                F.col("content").isNotNull() &
                (F.length(F.trim(F.col("content"))) > 0)
            )
        )
    )

    affected_row_count = (
        work_content_documents
        .filter(~F.col("is_content_present_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Required content validation: "
            "CONTENT_DOCUMENTS.content contains null or empty values."
        ),
        affected_row_count=affected_row_count,
        validation_code="NOT_NULL"
    )

    return work_content_documents

In [ ]:
def validate_content_id_uniqueness(
    work_content_documents,
    validation_results
):
    """
    Validate that content_id uniquely identifies each content document.
    """

    duplicate_content_ids = (
        work_content_documents
        .groupBy("content_id")
        .count()
        .filter(F.col("count") > 1)
        .select("content_id")
    )

    work_content_documents = (
        work_content_documents.alias("c")
        .join(
            duplicate_content_ids.alias("d"),
            F.col("c.content_id") == F.col("d.content_id"),
            "left"
        )
        .withColumn(
            "is_content_id_unique_passed",
            F.col("d.content_id").isNull()
        )
        .select(
            "c.*",
            "is_content_id_unique_passed"
        )
    )

    affected_row_count = (
        work_content_documents
        .filter(~F.col("is_content_id_unique_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Uniqueness validation: duplicate content_id values "
            "found in CONTENT_DOCUMENTS."
        ),
        affected_row_count=affected_row_count,
        validation_code="DUPLICATE"
    )

    return work_content_documents

## Determine Validation Outcome

Combines all validation flags into a final row-level result and returns the content documents that passed every validation rule.

In [ ]:
def determine_validation_outcome(
    work_content_documents
):
    """
    Combine all validation flags and returns only content documents
    that passed every validation rule.
    """

    work_content_documents = (
        work_content_documents
        .withColumn(
            "is_validation_passed",
            get_validation_condition(work_content_documents)
        )
    )

    valid_content_documents = (
        work_content_documents
        .filter(F.col("is_validation_passed"))
    )

    return (
        work_content_documents,
        valid_content_documents
    )

## Reconcile Row Counts

Compares the number of supported Bronze files from the source execution with the number of validated content documents eligible for Silver promotion.

In [ ]:
def reconcile_content_row_counts(
    valid_content_documents,
    source_container,
    source_execution_step_id,
    supported_extensions
):
    """
    Reconcile supported Bronze source files against validated
    content documents prepared for Silver promotion.
    """

    source_row_count = (
        spark.table("bronze.content_files")
        .filter(
            (F.col("source_container") == source_container) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .withColumn(
            "file_extension",
            F.lower(
                F.regexp_extract(
                    F.col("file_name"),
                    r"(\.[^.]+)$",
                    1
                )
            )
        )
        .filter(
            F.col("file_extension").isin(supported_extensions)
        )
        .count()
    )

    target_row_count = valid_content_documents.count()

    return [
        {
            "metric_name": "ROW_COUNT",
            "reconciliation_key": "CONTENT_DOCUMENTS",
            "reconciliation_side": "SOURCE",
            "metric_value_bigint": source_row_count
        },
        {
            "metric_name": "ROW_COUNT",
            "reconciliation_key": "CONTENT_DOCUMENTS",
            "reconciliation_side": "TARGET",
            "metric_value_bigint": target_row_count
        }
    ]

## Prepare Silver Content

Removes temporary validation flags from valid content documents and adds Silver execution lineage and audit metadata before promotion.

In [ ]:
def prepare_silver_content(
    valid_content_documents,
    execution_step_id,
    created_at,
    created_by
):
    """
    Remove temporary validation flags and adds Silver lineage
    and audit metadata.
    """

    validation_flag_columns = [
        column_name
        for column_name in valid_content_documents.columns
        if column_name.startswith("is_")
        and column_name.endswith("_passed")
    ]

    return (
        valid_content_documents
        .drop(
            *validation_flag_columns,
            "is_validation_passed"
        )
        .withColumn(
            "execution_step_id",
            F.lit(execution_step_id).cast("long")
        )
        .withColumn(
            "created_at",
            F.lit(created_at)
        )
        .withColumn(
            "created_by",
            F.lit(created_by)
        )
    )

## Write Valid Content to Silver

Replace the selected source container's Silver scope with validated content documents.

In [ ]:
def write_silver_content(
    silver_content_documents,
    source_container
):
    """
    Publish validated content to Silver using scoped snapshot replacement.
    """

    (
        silver_content_documents.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_container = '{source_container}'"
        )
        .saveAsTable("silver.content_documents")
    )

## Determine Process Status

Evaluates validation and reconciliation results to determine whether the process completed cleanly or requires observation.

In [ ]:
def determine_process_status(
    validation_results,
    reconciliation_results
):
    """Determine whether validation or reconciliation produced observations.

    Return the Boolean has_observations, not a status code.
    """

    has_validation_issues = any(
        notebook_output["affected_row_count"] > 0
        for notebook_output in validation_results
    )

    reconciliation_by_key = {}

    for notebook_output in reconciliation_results:
        key = notebook_output["reconciliation_key"]
        side = notebook_output["reconciliation_side"]

        reconciliation_by_key.setdefault(key, {})[side] = (
            notebook_output["metric_value_bigint"]
        )

    has_reconciliation_issues = any(
        values.get("SOURCE") != values.get("TARGET")
        for values in reconciliation_by_key.values()
    )

    return (
        has_validation_issues
        or has_reconciliation_issues
    )

## Execute Validation Process

Orchestrates the content validation workflow from Work loading through validation, reconciliation, Silver preparation, promotion, and final process status evaluation.

In [ ]:
created_at = datetime.now(timezone.utc)

validation_results = []

In [ ]:
work_content_documents = load_work_content(
    source_container=p_source_container
)

work_content_documents = validate_required_content_id(
    work_content_documents=work_content_documents,
    validation_results=validation_results
)

work_content_documents = validate_required_content(
    work_content_documents=work_content_documents,
    validation_results=validation_results
)

work_content_documents = validate_content_id_uniqueness(
    work_content_documents=work_content_documents,
    validation_results=validation_results
)

work_content_documents, valid_content_documents = (
    determine_validation_outcome(
        work_content_documents=work_content_documents
    )
)

reconciliation_results = reconcile_content_row_counts(
    valid_content_documents=valid_content_documents,
    source_container=p_source_container,
    source_execution_step_id=p_source_execution_step_id,
    supported_extensions=supported_extensions
)

silver_content_documents = prepare_silver_content(
    valid_content_documents=valid_content_documents,
    execution_step_id=p_execution_step_id,
    created_at=created_at,
    created_by=p_created_by
)

write_silver_content(
    silver_content_documents=silver_content_documents,
    source_container=p_source_container
)

has_observations = determine_process_status(
    validation_results=validation_results,
    reconciliation_results=reconciliation_results
)

## Output

Returns validation results, reconciliation metrics, and the final observation status to the calling pipeline.

In [ ]:
notebook_output = {
    "validation_results": validation_results,
    "reconciliation_results": reconciliation_results,
    "has_observations": has_observations
}

mssparkutils.notebook.exit(
    json.dumps(notebook_output)
)